In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from clawpack.pyclaw import Solution
from clawpack.visclaw import gridtools  # would work with v5.7.0

In [ ]:
frame = 15
h_ref = 100
sol_mbHSGN = Solution(frame, read_aux=True, path='./_output_fixed_grid_mbHSGN', file_format='binary')
sol_HSGN= Solution(frame, read_aux=True, path='./_output_fixed_grid_HSGN', file_format='binary')
sol_SWE = Solution(frame, read_aux=True, path='./_output_fixed_grid_SWE', file_format='binary')
sol_SGN = Solution(frame, read_aux=True, path='./_output_fixed_grid_SGN', file_format='binary')

time = sol_mbHSGN.t

In [ ]:
#Read npz data
dict_Euler = np.load('linear_2d_wave_data.npz')

In [ ]:
frames_Euler = dict_Euler['frames']
times_Euler = dict_Euler['times']
X_Euler = dict_Euler['X']
Y_Euler = dict_Euler['Y']
h_Euler = dict_Euler['h']
g_Euler = dict_Euler['g']

In [ ]:
#Get the closest time index
time_index = (np.abs(times_Euler - time)).argmin()

In [ ]:
h_mbHSGN = gridtools.grid_output_2d(sol_mbHSGN, 0, X_Euler, Y_Euler)
h_HSGN = gridtools.grid_output_2d(sol_HSGN, 0, X_Euler, Y_Euler)
h_SWE = gridtools.grid_output_2d(sol_SWE, 0, X_Euler, Y_Euler)
h_SGN = gridtools.grid_output_2d(sol_SGN, 0, X_Euler, Y_Euler)

In [ ]:
#Plot 3 different solutions at final time
fig, ax = plt.subplots(1, 4, figsize=(15,4))
vlims = [-0.05, 0.05]
colormap = "RdBu_r"
axs = ax.ravel()
axs[0].contourf(X_Euler, Y_Euler, frames_Euler[time_index,0,:,:], levels=50, cmap=colormap, vmin=vlims[0], vmax=vlims[1])
axs[0].set_title('Linear Free Surface Euler Equations')
axs[2].contourf(X_Euler, Y_Euler, h_SGN-100, levels=50, cmap=colormap, vmin=vlims[0], vmax=vlims[1])
axs[2].set_title('SGN'+r'')
axs[3].contourf(X_Euler, Y_Euler, h_HSGN-100, levels=50, cmap=colormap, vmin=vlims[0], vmax=vlims[1])
axs[3].set_title('HSGN')
axs[1].contourf(X_Euler, Y_Euler, h_SWE-100, levels=50, cmap=colormap, vmin=vlims[0], vmax=vlims[1])
axs[1].set_title('SWE')

axs[0].set_ylabel('y (m)')


for i in range(4):
    axs[i].set_xlabel('x (m)')
    axs[i].set_xlim([0, 5000])
    axs[i].set_ylim([0, 5000])

for i in range(1,4):
    #Remove y tick labels for all but first plot
    axs[i].set_yticklabels([])
    
plt.tight_layout()
plt.suptitle(f'Water surface elevation at t={time:.2f} s', y=1.02, fontsize=16)
#Save as high quality .png
plt.savefig('comparison_solutions.png', bbox_inches='tight', dpi=300)

### Interpolate at a a diagonal slice through the domain

In [ ]:
from scipy.interpolate import RegularGridInterpolator

In [ ]:
h_Euler_line = frames_Euler[time_index,0,:,:]
interp_Euler = RegularGridInterpolator((X_Euler[:,0], Y_Euler[0,:]), h_Euler_line)
num_points = 500
x_line = np.linspace(0, 4000, num_points)
y_line = np.linspace(0, 4000, num_points)
points_line = np.array([[x_line[i], y_line[i]] for i in range(num_points)])
h_Euler_diag = interp_Euler(points_line)   
h_ref = 100 #Plotting surface elevation

interp_SGN = RegularGridInterpolator((X_Euler[:,0], Y_Euler[0,:]), h_SGN)
h_SGN_diag = interp_SGN(points_line)
h_SGN_diag -= h_ref

interp_mbHSGN = RegularGridInterpolator((X_Euler[:,0], Y_Euler[0,:]), h_mbHSGN)
h_mbHSGN_diag = interp_mbHSGN(points_line)
h_mbHSGN_diag -= h_ref

interp_SWE = RegularGridInterpolator((X_Euler[:,0], Y_Euler[0,:]), h_SWE)
h_SWE_diag = interp_SWE(points_line)
h_SWE_diag -= h_ref

interp_HSGN = RegularGridInterpolator((X_Euler[:,0], Y_Euler[0,:]), h_HSGN)
h_HSGN_diag = interp_HSGN(points_line)
h_HSGN_diag -= h_ref

#Plot diagonal slice
plt.figure(figsize=(12,4))
plt.plot(x_line, h_Euler_diag, 'k-', label='Linear Free Surface Euler (reference)')
plt.plot(x_line, h_SGN_diag, 'm-', label='SGN')
plt.plot(x_line, h_mbHSGN_diag, 'g-', label='HSGN')
plt.plot(x_line, h_SWE_diag, 'b-', label='SWE')
plt.xlabel(r'$x = y (m)$')
plt.ylabel(r'$\eta (m)$')
plt.title(f'Water surface elevation along diagonal at $t=${time:.2f} $s$')
plt.legend(loc='upper left')
#Save as pdf
plt.savefig('comparison_solutions_diagonal.pdf', bbox_inches='tight')